# 03 — SHAP Analysis: Model Interpretability

SHAP (SHapley Additive exPlanations) values decompose each prediction into per-feature contributions.  
This is critical for credit risk — regulators (SR 11-7, ECOA) require explainability of credit decisions.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap


def find_repo_root():
    """Locate the repository whether Jupyter starts here or in notebooks/."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pipeline").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the GBM_cs repository")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline import config  # noqa: E402

GOLD_DIR = config.gold_dir()
CHAMPION_DIR = config.champion_dir()
model = joblib.load(config.model_path(CHAMPION_DIR))
calibrator_path = CHAMPION_DIR / "calibrator.joblib"
calibrator = joblib.load(calibrator_path) if calibrator_path.exists() else None

with open(GOLD_DIR / "feature_metadata.json") as f:
    meta = json.load(f)

feature_cols = meta["feature_columns"]
test = pd.read_parquet(GOLD_DIR / "features_test.parquet")
X_test = test[feature_cols]
y_test = test["default"]

sample_size = min(5_000, len(X_test))
sample_idx = np.random.RandomState(42).choice(
    len(X_test), size=sample_size, replace=False
)
X_sample = X_test.iloc[sample_idx]

print(f"Computing SHAP values for {len(X_sample):,} samples...")

In [ ]:
# Compute raw-model SHAP values. Normalize binary-class output across SHAP versions.
explainer = shap.TreeExplainer(model)
raw_shap_values = explainer.shap_values(X_sample)
if isinstance(raw_shap_values, list):
    shap_values = np.asarray(raw_shap_values[-1])
else:
    shap_values = np.asarray(raw_shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, -1]

base_value = float(np.asarray(explainer.expected_value).reshape(-1)[-1])
assert shap_values.shape == X_sample.shape

## 1. SHAP Summary Plot — Global Feature Importance

In [ ]:
# Beeswarm plot — shows direction and magnitude of each feature's impact
shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
plt.title("SHAP Summary — Top 20 Features", fontweight='bold')
plt.tight_layout()
plt.show()

## 2. SHAP Bar Plot — Mean Absolute Impact

In [ ]:
# Bar plot — mean |SHAP value| per feature
shap.summary_plot(shap_values, X_sample, plot_type="bar", max_display=20, show=False)
plt.title("Mean |SHAP Value| — Feature Importance Ranking", fontweight='bold')
plt.tight_layout()
plt.show()

## 3. SHAP Dependence Plots — Key Features

In [ ]:
# Dependence plots for top 4 features
top_4 = pd.DataFrame({'feature': feature_cols, 'importance': np.abs(shap_values).mean(axis=0)}) \
    .sort_values('importance', ascending=False).head(4)['feature'].tolist()

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, feat in enumerate(top_4):
    shap.dependence_plot(feat, shap_values, X_sample, ax=axes[i], show=False)
    axes[i].set_title(feat, fontweight='bold')

plt.suptitle("SHAP Dependence Plots — Top 4 Features", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Individual Explanation — Waterfall Plot

Show how a single credit decision was made (adverse action explanation).

In [ ]:
# Explain the highest-risk applicant in the sample.
raw_scores = model.predict_proba(X_sample)[:, 1]
scores = calibrator.predict(raw_scores) if calibrator is not None else raw_scores
applicant_idx = int(np.argmax(scores))
score = float(scores[applicant_idx])
if score < config.APPROVE_THRESHOLD:
    decision = "APPROVE"
elif score < config.REVIEW_THRESHOLD:
    decision = "REVIEW"
else:
    decision = "DECLINE"

actual = "DEFAULT" if y_test.iloc[sample_idx[applicant_idx]] == 1 else "PAID"
score_label = "Calibrated PD" if calibrator is not None else "Raw model score"
print(f"{score_label}: {score:.4f} ({decision})")
print(f"Actual outcome: {actual}\n")

# SHAP decomposes the raw LightGBM output. Calibration changes the PD scale,
# but it does not change the feature contributions or their ordering.
shap_explanation = shap.Explanation(
    values=shap_values[applicant_idx],
    base_values=base_value,
    data=X_sample.iloc[applicant_idx],
    feature_names=feature_cols,
)
shap.waterfall_plot(shap_explanation, max_display=15, show=False)
plt.title(f"Feature Contributions — {decision.title()} Applicant", fontweight="bold")
plt.tight_layout()
plt.show()